# DEMONEXT Startup/Shutdown Sandbox

This notebook is a sandbox for learning how to startup and shutdown the DEMONEXT systems.

Unlike the main DEMONEXT sandbox notebook, this one uses the `demonext` classes.

## WARNING: READ THIS NOW!

**Never, Ever, Run All in this notebook at once**  

This was designed to break the startup and shutdown processes into pieces and document them. Do not run this
all at once.

In [2]:
import sys
import os
import time
import math
import glob
import datetime
from datetime import date, timedelta

# Windows Component Object Model (COM) client module.

from win32com.client import Dispatch

# modules we need from anaconda

import numpy as np

# FITS writing and handling

from astropy.io import fits

# astropy units, time, and coordinate functions go here

import astropy.units as u

# we use pathlib for platform-agnostic path handling 

from pathlib import Path

# we use YAML for runtime configuration files

import yaml

# We use logging for runtime logs

import logging

## demonext module

In [5]:
import demonext
from demonext import config, pdu

## Convenience boolean translations

Dictionaries to translate booleans into human-readable form.

In [8]:
# Boolean state convenience translation dictionaries

OnOff = {True:'On',False:'Off'}
YesNo = {True:'Yes',False:'No'}

In [10]:
def getProcList():
    from win32com.client import GetObject
    wmi = GetObject("winmgmts:")
    procs = wmi.InstancesOf("win32_process") 
    procList = []
    for proc in procs:
        procList.append(proc.Name)
    return procList

## Load the runtime configuration file

DEMONEXT uses a YAML-formatted runtime configuration file kept in a common "flight" directory.

In [13]:
# platform-agnostic path definition relative to home

configDir = Path.home() / ".demonext/config"
configFile = "demonext.txt"

cfgFile = str(Path.home() / configDir / configFile)

# read by instantiating a demonext Config class

try:
    cfg = config.Config(cfgFile)
except Exception as exp:
    print(f"ERROR: (Config) - {exp}")

## Start the runtime logger

DEMONEXT uses the Python logging facility for runtime engineering logs.  The default directory is
obtained from the runtime configuration file loaded above. Engineering logs are have names `engCCYYMMDD.txt`,
where `CCYYMMDD` is the observing day, obtained using the `demonext.obsDate()` method.

In [16]:
logDir = demonext.homePath(cfg.config["directories"]["LogDir"])

logFile = str(Path(logDir) / f"eng{demonext.obsDate()}.txt")

logging.basicConfig(filename=logFile,
                    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
                    filemode="a",
                    level=logging.INFO)

# ID for log entries is "startup"
logger = logging.getLogger("startup")

msg = "Started the DEMONEXT startup procedure"
logger.info(msg)
print(msg)

Started the DEMONEXT startup procedure


## AC power control

DEMONEXT uses a Raritan PXO 4-outlet power-distribution unit (PDU) to control AC power to essential systems.
Instantiate an PDU instance, using the PDU configuration file passed in the runtine configuration file. 

In [19]:
# retrieve the pdu config dictionary from the config file

try:
    pduInfo = cfg.config["pdu"]
except Exception as exp:
    print(f"ERROR: {exp}")

# Instantiate a Raritan PDU interface instance as "power" for AC power
# control using the runtime configuration file given as the "config"
# entry in pduInfo.  Paths are relative to home, but might be absolute.

try:
    pdu = pdu.RaritanPDU(demonext.homePath(pduInfo["config"]))
    print("Connected to the DEMONEXT PDU")
except Exception as exp:
    print(f"ERROR: {exp}")

Connected to the DEMONEXT PDU


### Power on the main observatory systems

Outlets we want to control at startup:
 * `CCD` - science CCD package: the FLI CCD camera, FLI filter wheel, and telescope Hedrick focuser
 * `Telescope` - the SiTech servo controller for the telescope mount
 
Note that the outlet IDs are case-insensitive, the actual device-level IDs are all lowercase, and the
demonext PDU class functions force outlets IDs to lowercase for all validity tests.  Also note that
labels for the outlets on the PDU were set to lowercase versions of these.

Actions for each outlet:
 * If the outlet is off, we power it on
 * If the outlet is on, we power cycle it
 
At the end we set the `allOn` boolean True if all systems are on, False if one or more are off.

In [22]:
logger.info("Subsystem power-up")

outlets = ['CCD','Telescope']

numOn = 0
for outlet in outlets:
    if pdu.isOn(outlet):
        print(f"{outlet} power is ON - cycling power (requires 10s)...")
        try:
            pdu.cycle(outlet)
        except Exception as exp:
            msg = f"ERROR: Cannot power cycle the {outlet} outlet: {exp}"
            print(msg)
            logger.exception(msg)
    else:
        print(f"Switching {outlet} ON ...")
        try:
            pdu.switch(outlet,"on")
        except Exception as exp:
            msg = f"ERROR: Cannot switch on the {outlet} outlet: {exp}"
            print(msg)
            logger.exception(msg)
            
    if pdu.isOn(outlet):
        numOn += 1
        
print(f"Done: Startup power status:")
for outlet in outlets:
    print(f"  {outlet} is {pdu.OnOff[pdu.isOn(outlet)]}")
if numOn < len(outlets):
    allOn = False
    logger.error(f"Only {numOn} of {len(outlets)} required subsystems powered On")
else:
    allOn = True
    logger.info("All subsystems powered on")


Switching CCD ON ...
Switching Telescope ON ...
Done: Startup power status:
  CCD is On
  Telescope is On


## Observatory Systems startup

We use the following ASCOM interfaces
 * `MaxIm.Application` - communicates with the MaxIM DL application class
 * `MaxIm.CCDCamera` - Operate the science and guide cameras through MaxIm DL Pro 7
 * `SiTech.Telescope` - Operate the telescope mount SiTech ServerController II via the PlaneWave STI app
 * `ASCOM.PWI3.Focuser` - Operate the PlaneWave Hedrick focuser (science camera) using the PlaneWave PWI3 app
 * `PlaneWave.AutoFocus` - access the PlaneWave telescope autofocus system through the PWI3 app

### Applications we need running

We run these applications, **in order**
 * `SitechExe.exe` - PlaneWave STI version 1.4.3, must execute by hand (not sure why, but we do), must run FIRST before MaxIm DL
 * `MaxIm DL Pro 7` - This is started automatically when we instantiate ASCOM object `MaxIm.Application` with the win32 COM client.  Must start after PlaneWave STI otherwise it cannot connect to the telescope mount interface without an app restart
 * `PlaneWave Interface v3` (PWI3) - This is started automatically inconified when we first instantiate the focuser through ASCOM (`ASCOM.PWI3.Focuser`)

### See which processes are running

Use the demonext `procList()` method to get the process list and check if any of the processes are running.


In [25]:
# DEMONEXT apps we need in order of startup

appList = ["SitechExe.exe","MaxIm_DL.exe","PWI3.exe"]

# list of all processes running on the system

procList = demonext.procList()

# procList = getProcList()

# Report on which processes are running before startup

print("Pre-startup observatory app status:")
for app in appList:
    if app in procList:
        print(f"  {app} is running")
    else:
        print(f"  {app} is not running")

Pre-startup observatory app status:
  SitechExe.exe is not running
  MaxIm_DL.exe is not running
  PWI3.exe is not running


### Start the PlaneWave STI app

For reasons that are still not clear, even to PlaneWave, instantiating the `SiTech.Telescope` ASCOM class
does not start the `SitechExe.exe` executable for the STI app.

So, we start it up by hand. To make this easier, we need to put the full path to the `SitechExe.exe` binary
into the Windows PATH.  Otherwise we get into having to use the full path below.  If the full path is
required, because Windows paths have backslashes as path separators you need to use raw string literal
(r"") to construct the path.

We use `os.startfile()` which is safer than `os.exec()`.

Wait 5 seconds for it to launch and finish its internal startup before attempting to connect to it
with a demonext Telescope class (not done in this notebook).

In [28]:
# full path to the SitechExe.exe executable.  Use if not yet in the Windows PATH
#
# stiExe = r"C:\Program Files (x86)\Common Files\ASCOM\Telescope\SiTech\SitechExe.exe"
#
# short version if we have installed the observatory apps in the Window PATH

logger.info("PlaneWave STI app startup")

stiExe = "SitechExe.exe"

procList = getProcList()

if stiExe in procList:
    print(f"Warning: {stiExe} instance is already running")
    logger.warning(f"{stiExe} instance already running")
else:
    print(f"Starting {stiExe}...")
    try:
        os.startfile(stiExe)
        time.sleep(5)
    except Exception as exp:
        msg = f"Cannot start {stiExe}: {exp}"
        print(msg)
        logger.exception(msg)

# verify it is running by connecting using ASCOM and retrieving the description

SiTech = Dispatch("SiTech.Telescope")
SiTech.Connected = True
if (SiTech.Connected):
    msg = f"Done: Connected the {SiTech.Description}"
    print(msg)
    logger.info(msg)
else:
    msg = f"ERROR: {stiExe} connection attempt failed!"
    print(msg)
    logger.exception(msg)

Starting SitechExe.exe...
Done: Connected the DEMONEXT SiTech Servo Controller at SRO


### Start MaxIm DL

Start the MaxIm DL `Application` and `CCDCamera` interfaces.  If MaxIm is not running, the first `Dispatch()`
launches the main app, the second `Dispatch()` launches the MaxIm CCD camera tool, hence the 
5s waits after each to give the apps time to launch and settle.

Set the `MaxIm.LockApp` property True - this ensures that the MaxIM app will keep running even if all of
the COM objects have closed.

We then enable the camera link, which instructions the MaxIm CCDCamera tool to try to 
connect to both cameras, science and guider.  If either is missing it will fail.

Final step is to disable auto shutdown (`DisableAutoShutdown=True`) so that MaxIm keeps the cameras running 
even if we have to stop and restart ASCOM links.

In [31]:
logger.info("MaxIm DL startup")
print("Starting MaxIm DL...")

try:
    MaxIm = Dispatch("MaxIm.Application")
    time.sleep(5)
    msg = f"MaxIm DL application started"
    print(f"  {msg}")
    logger.info(msg)
except Exception as exp:
    msg = f"ERROR: Cannot connect MaxIm DL: {exp}"
    print(msg)
    logger.exception(msg)

MaxIm.LockApp = True

try:
    MaxIm.TelescopeConnected = True
    msg = f"Telescope drives connected to MaxIm DL"
    print(f"  {msg}")
    logger.info(msg)
except Exception as exp:
    msg = f"ERROR: Cannot connect MaxIm DL to the telescope: {exp}"
    print(msg)
    logger.exception(msg)

# start the camera

try:
    MaxCam = Dispatch("MaxIm.CCDCamera")
    time.sleep(5)
    msg = f"MaxIm DL CCDCamera control panel launched"
    print(f"  {msg}")
    logger.info(msg)
except Exception as exp:
    msg = f"Cannot launch MaxIm DL camera control panel: {exp}"
    print(msg)
    logger.exception(msg)
    print(f"Check physical connection, especially the USB to the guide camera")

# try to connect the cameras

try:
    MaxCam.LinkEnabled = True
    msg = f"Cameras linked to MaxIm DL"
    print(f"  {msg}")
    logger.info(msg)
except Exception as exp:
    msg = f"Cannot connect cameras to MaxIM DL: {exp}"
    print(msg)
    logger.exception(msg)

# This makes sure MaxIm does not shutdown cameras if all COM objects terminate

MaxCam.DisableAutoShutdown = True

msg = "MaxIm DL startup complete"
print(f"Done: {msg}")
logger.info(msg)

Starting MaxIm DL...
  MaxIm DL application started
  Telescope drives connected to MaxIm DL
  MaxIm DL CCDCamera control panel launched
  Cameras linked to MaxIm DL
Done: MaxIm DL startup complete


### Start the PlaneWave Hedrick Focuser

The Hedrick Focuser at the CDK20 cassegrain focus is controlled using the PlaneWave `PWI3` app through
the `ASCOM.PWI3.Focuser` object.  Invoking this opens PWI3 minimized on the task bar.

We also start the PlaneWave AutoFocus application. Although it cannot be tested except on-sky, we
can verify it is available and correctly installed.

In [34]:
logger.info("PWI3 Focuser and AutoFocus system startup")
print("Connecting Focuser and AutoFocus systems...")

try:
    Focuser = Dispatch("ASCOM.PWI3.Focuser")
    msg = "Started focuser"
    print(f"  {msg}")
    logger.info(msg)
except Exception as exp:
    msg = f"ERROR: Cannot start PWI3 Focuser app: {exp}"
    print(msg)
    logger.exception(msg)
    
try:
    AutoFoc = Dispatch("PlaneWave.AutoFocus")
    msg = "Started PlaneWave AutoFocus"
    print(f"  {msg}")
    logger.info(msg)
except Exception as exp:
    msg = f"ERROR: Cannot start PWI3 AutoFocus: {exp}"
    print(msg)
    logger.exception(msg)

# try to connect

try:
    Focuser.Connected = True
    msg = f"Done: Connected to {Focuser.Description}, Focuser Link Enabled? {YesNo[Focuser.Link]}"
    print(msg)
    logger.info(msg)
except Exception as exp:
    msg = f"ERROR: Cannot connect focuser: {exp}"
    print(msg)
    logger.exception(msg)

time.sleep(5.0)

print(f"Ambient Temperature: {AutoFoc.GetTemperatureByName('Ambient.EFA'):.2f} C")
print(f"Primary Mirror Temperature: {AutoFoc.GetTemperatureByName('Primary.EFA'):.2f} C")

Connecting Focuser and AutoFocus systems...
  Started focuser
  Started PlaneWave AutoFocus
Done: Connected to PlaneWave Focuser (PWI3), Focuser Link Enabled? Yes
Ambient Temperature: 21.20 C
Primary Mirror Temperature: 20.30 C


## Startup complete

All systems hould now be powered on, all apps running, and ASCOM connections established to all relevant
observatory components.

At this point you could begin data acquisition operations with the telescope system.

The shutdown procedure follows

## Observatory system shutdown

Order of shutdown:
 * Disconnect all active ASCOM links
 * Shutdown the Windows apps (MaxIm, STI, PWI3)
 * Power off the CCD and Telescope systems
 
These steps assume you have already parked the telescope

In [38]:
# ID for log entries is "shutdown"

logger = logging.getLogger("shutdown")

logger.info("Started the DEMONEXT shutdown procedure")

# DEMONEXT apps

appList = ["SitechExe.exe","MaxIm_DL.exe","PWI3.exe"]

# update the process list since startup

procList = demonext.procList()

# Report on which processes are running now

print("Pre-shutdown observatory app status:")
for app in appList:
    if app in procList:
        print(f"  {app} is running")
    else:
        print(f"  {app} is not running")

Pre-shutdown observatory app status:
  SitechExe.exe is running
  MaxIm_DL.exe is running
  PWI3.exe is running


## Disconnect ASCOM links

Need to make sure ASCOM instances are cleanly detached, or you can get stuck processes
that lead to strange errors ("why is it not connecting? It was just connected?")

In [41]:
if Focuser.Connected:
    msg = f"Disconnecting Hedrick Focuser (PWI3)"
    print(msg)
    logger.info(msg)
    try:
        Focuser.Connected = False
    except Exception as exp:
        logger.exception(exp)
        
if MaxIm.TelescopeConnected: 
    msg = "Disconnecting the telescope from MaxIm"
    print(msg)
    logger.info(msg)
    try:
        MaxIm.TelescopeConnected = False
        time.sleep(2)
    except Exception as exp:
        logger.exception(exp)
        
if MaxCam.LinkEnabled:
    msg = "Disconnecting the science and guide cameras from MaxIm"
    print(msg)
    logger.info(msg)
    try:
        MaxCam.LinkEnabled = False
        time.sleep(2)
    except Exception as exp:
        logger.exception(exp)

# Unlock MaxIm DL - allows app to close when all COM object close

MaxIm.LockApp = False

if SiTech.Connected:
    msg = "Disconnecting the SiTech telescope mount controller (PlaneWave STI)"
    print(msg)
    logger.info(msg)
    try:
        SiTech.Connected = False
        time.sleep(2)
    except Exception as exp:
        print(exp)

# complete

print('Done: DEMONEXT subsystem ASCOM links disconnected.')

Disconnecting Hedrick Focuser (PWI3)
Disconnecting the telescope from MaxIm
Disconnecting the science and guide cameras from MaxIm
Disconnecting the SiTech telescope mount controller (PlaneWave STI)
Done: DEMONEXT subsystem ASCOM links disconnected.


## Take down Apps running on the system

Some wicked windows tricks here courtesy of too much time on stack overflow and other websites.
There is no one, best way to do things, but this seems to work robustly.

Uses the `win32com.client` class to access the
[Windows Management Instrumentation (WMI)](https://learn.microsoft.com/en-us/windows/win32/wmisdk/wmi-start-page) command-line tool `[winmgmts](https://learn.microsoft.com/en-us/windows/win32/wmisdk/winmgmt)`.  This lets us access the process table that would be returned, for example, by the windows shell `tasklist` command, in a machine-readable form we can quickly sift through with python.

Here we capture the names of all processes running on the system in `procList`, and loop through
a list of apps to terminate and if the app is in `procList` we execute `taskkill /f /im app.exe /t` to 
stop the app and all associated child processes.  This makes for a very clean shutdown of the observatory
apps.

In [44]:
# update the process list just in case...

procList = getProcList()

# only shutdown apps that are still running

for exe in ["MaxIm_DL.exe","PWI3.exe","SitechExe.exe"]:
    if exe in procList:
        msg = f"Shutting down {exe}..."
        print(msg)
        logger.info(msg)
        try:
            result = os.system(f"taskkill /f /im {exe} /t")
        except Except as exp: 
            msg = f"Could not shutdown {exe}: {exp}"
            print(msg)
            logger.exception(msg)
msg = "Done: all apps shutdown"
print(msg)
logger.info(msg)

Shutting down MaxIm_DL.exe...
Shutting down PWI3.exe...
Shutting down SitechExe.exe...
Done: all apps shutdown


### Power OFF the main observatory systems

Last step is to turn off power to the main observatory systems

Outlets we need to control at shutdown:
 * `CCD` - science CCD package: the FLI CCD camera, FLI filter wheel, and telescope Hedrick focuser
 * `Telescope` - the SiTech servo controller for the telescope mount
 
Note that the outlet IDs are case-insensitive, the actual device-level IDs are all lowercase, and the
demonext PDU class functions force outlets IDs to lowercase for all validity tests.  Also note that
labels for the outlets on the PDU were set to lowercase versions of these.

In [47]:
logger.info("Subsystem power-off")

outlets = ['CCD','Telescope']

numOn = 0
for outlet in outlets:
    print(f"Switching {outlet} OFF ...")
    try:
        pdu.switch(outlet,"off")
    except Exception as exp:
        msg = f"ERROR: Cannot switch off the {outlet} outlet: {exp}"
        print(msg)
        logger.exception(msg)
            
print(f"Done - shutdown power status:")
for outlet in outlets:
    print(f"  {outlet} is {pdu.OnOff[pdu.isOn(outlet)]}")
    
print(f"\nDone: DEMONEXT shutdown completed")

Switching CCD OFF ...
Switching Telescope OFF ...
Done - shutdown power status:
  CCD is Off
  Telescope is Off

Done: DEMONEXT shutdown completed
